# mawsons-chest example

This notebook demonstrates a minimal workflow for `mawsons-chest` using the `SeaIceFigureToolbox` class.

It is designed for direct daily CICE history files (`iceh.*.nc`) and optional daily NSIDC comparison.

In [ ]:
from pathlib import Path
import xarray as xr

from src.sea_ice_figure_toolbox import SeaIceFigureToolbox

## Initialise the toolbox

The defaults below follow the current Gadi-style directory structure used in your workflow. Override any path here if needed.

In [ ]:
tb = SeaIceFigureToolbox(
    cice_history_dir=Path("/g/data/gv90/da1339/cice-dirs/runs/free-slip-waves/history"),
    nsidc_daily_south_dir=Path("/g/data/gv90/da1339/SeaIce/NSIDC/G02202_V4/south/daily"),
    nsidc_daily_north_dir=Path("/g/data/gv90/da1339/SeaIce/NSIDC/G02202_V4/north/daily"),
    nsidc_cell_area_south=Path("/g/data/gv90/da1339/SeaIce/NSIDC/NSIDC0771/NSIDC0771_CellArea_PS_S25km_v1.1.nc"),
    nsidc_cell_area_north=Path("/g/data/gv90/da1339/SeaIce/NSIDC/NSIDC0771/NSIDC0771_CellArea_PS_N25km_v1.1.nc"),
    output_dir=Path("./figures"),
    animation_dir=Path("./animations"),
    sic_threshold=0.15,
)

## Choose a CICE day

In [ ]:
date_str = "1993-04-24"
ds = tb.load_cice_day(date_str)
dt_str = tb.cice_corrected_datestr(ds)

print("Requested file date:", date_str)
print("Corrected CICE date string:", dt_str)
ds

## Compute hemispheric sea-ice extent from CICE `aice`

In [ ]:
ext = tb.compute_cice_ice_extent(ds)

print("SH SIE:", ext["south"], ext["units"])
print("NH SIE:", ext["north"], ext["units"])
print("Threshold:", ext["threshold"])

## Compute hemispheric SIA, SIV, and aggregate SIT from `hi` and `aice`

In [ ]:
stats = tb.compute_cice_area_volume_thickness(ds)

print("SH SIA:", stats["south"]["SIA"], stats["SIA_units"])
print("SH SIV:", stats["south"]["SIV"], stats["SIV_units"])
print("SH SIT:", stats["south"]["SIT"], stats["SIT_units"])
print()
print("NH SIA:", stats["north"]["SIA"], stats["SIA_units"])
print("NH SIV:", stats["north"]["SIV"], stats["SIV_units"])
print("NH SIT:", stats["north"]["SIT"], stats["SIT_units"])

## Daily concentration figure with NSIDC southern contour

In [ ]:
fig = tb.plot_aice_day(
    date_str,
    add_nsidc_south=True,
    add_nsidc_north=False,
    show=False,
)

fig

## Daily thickness figure from model `hi`

In [ ]:
fig = tb.plot_hi_day(
    date_str,
    show=False,
)

fig

## Save figures to disk

In [ ]:
tb.plot_aice_day(
    date_str,
    add_nsidc_south=True,
    output_path=Path("./figures") / f"aice_{date_str}.png",
    show=False,
)

tb.plot_hi_day(
    date_str,
    output_path=Path("./figures") / f"hi_{date_str}.png",
    show=False,
)

## Optional: add a SIA timeseries inset

This builds simple daily time series from CICE and NSIDC and draws an inset on the `aice` figure.

In [ ]:
fig = tb.plot_aice_day(
    date_str,
    add_nsidc_south=True,
    add_sia_timeseries=True,
    ts_start="1993-04-01",
    ts_end="1993-04-30",
    show=False,
)

fig

## Optional: create an animation

This will render daily frames and combine them into a GIF or MP4.

In [ ]:
# Uncomment to build a GIF animation
# gif_path = tb.create_animation(
#     dt0_str="1993-04-01",
#     dtN_str="1993-04-10",
#     variable="aice",
#     add_sia_timeseries=True,
#     fps=4,
#     codec="gif",
# )
# print(gif_path)

## Notes

- The class assumes direct `iceh.*.nc` access rather than a prebuilt Zarr archive.
- The corrected CICE date is computed by subtracting one day from the in-file time coordinate.
- NSIDC daily metrics use the official NSIDC0771 cell-area files for the chosen hemisphere.
- The northern NSIDC path is already wired into the class, so once the files are in place the same notebook cells can be used for NH comparisons.